# Realtime 评估指南

<p align="center">
<img
  width="650"
  src="../images/realtime_eval_page_01_image_01.png"
  alt="第 1 页的示意图"
/>
</p>


## 简介

评估（evals）能把一个语音演示变成人们可以信赖的产品。“看起来还行”与“每天都能稳定工作”之间的差距，几乎总是由评估来弥补的。

本指南展示了如何通过逐步提升复杂度来评估语音系统：从简单开始（Crawl，爬），加入真实感（Walk，走），然后测试多轮对话（Run，跑）。在此过程中，你将学会构建让结果稳健可靠的三样东西：数据集、评分器（grader）和评估框架（eval harness），以及一个生产飞轮，让真实的失败案例转化为新的测试。

投入评估的团队交付到生产环境的速度可以快 **5–10 倍**，因为他们能够看到哪里出了问题、精确定位原因，并有把握地进行修复。


### Realtime 评估框架代码

如果你需要可运行的代码来构建评估框架，请先使用这个仓库文件夹：

- **GitHub 仓库路径：** [openai-cookbook/examples/evals/realtime_evals](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals)
- 其中包含每个成熟度阶段的完整参考框架：
  - [Crawl 框架（单轮回放）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/crawl_harness)
  - [Walk 框架（已保存音频回放）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/walk_harness)
  - [Run 框架（模型模拟的多轮对话）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/run_harness)

你可以让 Codex 指向你需要的框架，并让它根据你的数据和评分器进行适配。

## 第一部分：基础

### 1) 为什么 realtime 评估很难

Realtime 比文本更难，因为你要评分的是一个**流式交互**，它有两种输出：**助手做了什么**，以及**它听起来怎么样**。一个回复可能“内容正确”，但听起来仍然有问题。

#### 1.1 realtime 质量的两个维度

文本评估主要关注内容是否正确。Realtime 增加了第二个维度：音频质量。内容和音频可能各自独立地出问题，因此单一分数可能掩盖真实存在的问题。

**大多数 realtime 评估都可以归结为两个相互独立的维度：**

1. 内容质量：助手是否理解了用户并做了正确的事？包括正确性、有用性、工具选择、工具参数和指令遵循。

2. 音频质量：助手听起来是否可以接受？包括自然度、韵律、发音、稳定性，以及在噪声和采集不完美情况下的表现。



#### 1.2 难以调试

使用 **Responses API** 时，心智模型很简单：**请求进 → 响应出**。而使用 **Realtime API** 时，一个“轮次”是一条**拼接起来的流水线**。这种编排让语音应用易于构建，但对于评估来说，你必须记录各个阶段的日志，才能隔离故障并找到根本原因。

<p align="center">
<img
  width="700"
  src="../images/realtime_eval_page_02_image_01.png"
  alt="第 2 页的示意图"
/>
</p>


一个“轮次”是一连串事件（语音开始/结束 → 提交 → response.create → 音频增量 → 完成），任何一个阶段都可能出错。如果你把系统当作黑盒，就会去追查所谓的“模型问题”，而实际上它们是轮次检测、缓冲或工具集成方面的问题。

**示例：**

- 内容正确但体验糟糕：由于打断边界判断错误，用户插话（barge-in）时音频被截断。

- 回答“正确”但感觉很慢：延迟来自网络质量和轮次检测缓慢，而不是模型的推理。

你可以在[这里](https://platform.openai.com/docs/api-reference/realtime-server-events)了解 Realtime API 触发的各种事件。

#### 1.3 转录文本 ≠ 真实情况

在 realtime API 中，“用户说了什么”的真实依据是**实际的音频信号**（麦克风采集到的内容以及模型听到的内容）。转录文本并不是真实依据，它是模型对该音频做出的**解读**。由于受到**转录模型错误**的限制，它可能是错的。

**如果你把转录文本当作真实情况，评估结果可能会产生误导：**

- 误判失败：ASR 漏掉了一个数字，但模型听到了并正确调用了工具 → 你的 LLM 评分器却判定为“错误”。

- 误判通过：转录文本看起来很干净，但音频被截断，模型是猜的 → 你错过了真正的问题。

**最佳实践：**

- **改进转录：** 迭代优化转录[提示词](https://platform.openai.com/docs/guides/speech-to-text#prompting)，尝试不同的[模型](https://platform.openai.com/docs/guides/speech-to-text)，尝试不同的方法，例如[带外（oob）转录](https://cookbook.openai.com/examples/realtime_out_of_band_transcription)。

- **用转录文本实现规模化：** 大部分自动化评分基于**转录文本 + 追踪记录（trace）**运行。

- **在杂乱的真实数据上校准评分器：** 在**类似生产环境的、带噪声的转录文本**（而不是干净的文本）上迭代评分器，以免它们对 ASR 错误反应过度。

- **增加音频审核环节：** 对 **约 1–5%** 的会话进行端到端抽查。

<p align="center">
<img
  width="650"
  src="../images/realtime_eval_page_03_image_01.png"
  alt="第 3 页的示意图"
/>
</p>


## 第二部分：策略

### 2) Crawl / Walk / Run

如果团队一开始就从最难的设置入手——真实音频、多轮对话和真实工具——realtime 评估会让人不知所措。解决办法是**分步提升复杂度**。如果你的系统**连爬都不会，就不可能跑起来**。早期的评估应当足够简单，使失败可诊断、可复现，且迭代成本低。你可以在两个相互独立的维度上提升复杂度。

#### 2.1 隔离输入条件：干净音频 vs 生产音频

这个维度关乎模型听到了什么。通过控制输入音频的条件，你可以把模型智能方面的失败与语音感知方面的失败区分开来。

- **从合成音频开始 → 测试智能：**
    - 当你想衡量模型的推理和决策能力、而不希望音频差异干扰信号时，使用干净、合成、可复现的音频（例如 TTS）→ 有助于隔离意图路由、工具调用和指令遵循的问题

- **再过渡到带噪声的、类似生产环境的音频 → 测试音频感知：**
    - 一旦智能表现稳定，就引入类似生产环境的音频：压缩、回声、远场采集、背景噪声、犹豫/自我纠正。这可以测试当输入含糊、杂乱或部分丢失时，系统是否仍能正确运行 → 有助于衡量听错词的情况以及对声学变化的鲁棒性

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_04_image_01.png"
  alt="第 4 页的示意图"
/>
</p>


#### 2.2 隔离交互条件：单轮 vs 多轮

这个维度关乎你在评估什么：是评估下一个轮次，还是评估整个**对话**。

- **从单轮开始 → 测试核心能力：**
    - 当你想获得关于基础能力最干净的信号时，运行“一个请求 → 一个响应”：正确的意图路由、正确的工具选择、有效的参数以及基本的指令遵循。如果系统在这里都无法可靠地选出正确的工具或生成有效的 schema，那么评估更多轮次也无济于事。

- **再过渡到多轮 → 测试鲁棒性：**
    - 一旦单轮表现稳定，就过渡到多轮：系统必须在多个轮次中保持目标和约束、正确地编排工具调用顺序、从工具故障中恢复并处理用户的更正。多轮评估让你从轮次级别的正确性转向**整段对话级别的结果**：它是否完成了目标？用了多少轮？出现问题时是否干净利落地恢复了？

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_04_image_02.png"
  alt="第 4 页的示意图"
/>
</p>


单轮评估告诉你*能否赢下一场战斗*；多轮评估告诉你*能否赢得整场战争*。

#### 2.3 评估象限

使用一张 2x2 的评估地图：**向右** = 音频更真实，**向上** = 交互更真实。从左下角开始，每次只在一个维度上提升难度。

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_05_image_01.png"
  alt="第 5 页的示意图"
/>
</p>


**评估模式（复杂度递增）：**

1. Crawl（左下）：合成音频 + 单轮

2. Walk（向右移动）：真实带噪声音频 + 单轮

3. Run（向上移动）：合成音频 + 多轮模拟 

右上角（真实音频 + 完整多轮流程）是人工评估：像用户在生产环境中那样运行端到端会话。在整个项目生命周期中都要保留这一环节。

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_05_image_02.png"
  alt="第 5 页的示意图"
/>
</p>


**示例：**

用户：“把我的预订改到晚上 7 点。”

- **Crawl：** 输入“Change my reservation to 7pm”的确定性 TTS 音频，然后只对助手的下一个轮次评分：它应当路由到预订更新工具，并传入正确的 time=7pm（如果缺少必需的标识信息，则提出一个简洁的澄清问题）。
- **Walk：** 录制一段由真人麦克风说出的“Change my reservation to 7pm”，然后在电话带宽压缩和轻微背景噪声下回放同一句话；系统仍应听到“7pm”（而不是“7”或“7:15”），并产生同样正确的工具调用。
- **Run：** 由模型模拟用户说出“Change my reservation to 7pm”，然后模拟真实的后续对话（“是以 Minhajul 的名字订的今晚……其实改成 7:30 吧……等等，改成明天”），并注入一次工具错误；智能体应当只针对缺失的信息进行澄清，保持状态一致，干净利落地恢复，最后以一次反映最终预期结果的正确更新工具调用结束。

你可以在这里找到可以直接上手并加以改造的参考实现：[realtime eval start](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals)。

## 第三部分：三大构建模块

### 4) 数据：构建基准

#### 4.1 从“黄金”种子集（10–50 条）开始

覆盖那些绝对不能失败的流程：核心意图、必须正常工作的工具调用、升级转人工和拒绝行为。先快速生成，然后由人工审核其真实性和遗漏之处。

**目标是先开始，而不是追求完美。**

#### 4.2 为迭代而构建，而不只是追求数量

评估数据集的存在是为了推动迭代，而不是为了看起来规模庞大。这个循环本身才是产品：**运行评估 → 将失败定位到具体行为 → 只改动一处 → 重新运行 → 确认修复有效且没有引入回归**。如果一个基准能让这个循环快速、可复现且易于诊断，它就是“好”的基准。

这需要的是覆盖面，而不是原始数量：你需要涵盖真实的用户行为以及导致生产故障的特定边界情况。仅靠规模无法暴露脆弱性；合适的覆盖面才可以。 

覆盖面还必须保持平衡。对于每种行为，都要同时包含正例（系统应当做 X）和反例（系统不应当做 X）。没有反例，你就是在奖励投机取巧。

>**客户案例：** 一个团队构建了一个语音客服机器人，并针对 "escalate_to_human" 工具调用进行了大力优化。他们在离线评估中的升级转人工得分达到了 98%。但在内部试用时，机器人几乎对所有情况都转人工。根本原因在于数据集不平衡。他们有大量“必须转人工”的案例，却几乎没有“不应转人工”的案例，于是模型学会了一条捷径：只要不确定就转人工。 

最后，你必须对数据进行精确的标记，以支持细粒度的评估。这些标签应当提供足够的细节，让你能从“分数下降了”这样笼统的观察，深入到具体的根本原因，例如**“该意图在这些音频条件和这条政策边界下会失败”。**

标签示例：

- 意图、预期结果、音频条件、语言和预期的工具调用。带标签的数据让团队能够进行细粒度评估，从而加快迭代循环。

<p align="center">
<img
  width="1200"
  src="../images/realtime_eval_page_07_image_01.png"
  alt="第 7 页的示意图"
/>
</p>


#### 4.3 从生产故障中扩充

离线评估是快速迭代的手段，但也很容易被“用到过时”。如果你一直针对一个固定的基准进行优化，**分数可能会上升，而真实质量（现实情况）却停滞不前**，因为用户会做出你的数据集没有覆盖的行为。

<p align="center">
<img
  width="650"
  src="../images/realtime_eval_page_07_image_02.png"
  alt="第 7 页的示意图"
/>
</p>


运作模式是一个循环：由生产环境扩充基准。出现了一个新的失败，你复现它、标记它，然后把它加进去。随着时间推移，你的离线测试集应当随产品一起成长。

**一种简单的管理方式是使用三个集合：**

- **回归测试集：** 你已经修复过的困难案例。每次修改提示词、模型和工具时都要运行。这是你“不能搞坏”的契约。

- **滚动发现集：** 来自生产环境的新鲜失败案例和险些失败的案例。在这里你可以了解自己遗漏了什么，以及下一步该优先处理什么。如果它们触发了失败模式，就将其提升到**你的离线数据集中。团队通常通过以下方式填充这个集合：**

    - 运行在线评分器直接捕获失败，和/或

    - 监控代理指标（延迟、工具错误率、转人工率、重试次数），并在指标漂移时对数据进行抽样。
- **保留集（holdout set）：** 离线测试中保持不动的一个子集，偶尔运行一次，用于检测对基准的过拟合。如果测试分数上升而保留集分数持平，说明你是在为考试而训练。

<p align="center">
<img
  width="650"
  src="../images/realtime_eval_page_08_image_01.png"
  alt="第 8 页的示意图"
/>
</p>


### 5) 评分器

评分器是你的**测量仪器**。它们把杂乱的实时语音会话转化为**可以信赖的信号**。

#### 5.1 人工审查（杠杆最高）

人工审查 = 收听真实音频 + 端到端阅读完整的追踪记录。这是建立产品直觉、发现用户能立即察觉的问题的最快方式。自动化评估告诉你能衡量什么；人工审查告诉你应该衡量什么。

**自动化常常低估、但用户能立刻感受到的问题：**

- 轮流发言失败：尴尬的停顿、双方同时说话、模型打断用户。

- 节奏与韵律：模型语速过快/过慢、啰嗦、语调平淡、不稳定、“机器人式的客气”。

- 转录文本不匹配：ASR 延迟/丢失/规范化问题 → 最终你评分的对象是错的。

- 评估系统自身的 bug：黄金集覆盖缺失、预期标注错误、评分器系统性地过严/过宽。 

> **客户案例：** 一家大公司让高管们每天花 **约 3 小时** 专门收听会话并浏览追踪记录。他们发现了一些“隐藏”的问题——过早截断、虚假打断、别扭的韵律——这些问题原本会在离线评估中被轻易放过。

#### 5.2 自动化评分器

人工无法规模化。没有自动化，回归问题就会悄悄溜过，所谓的“改进”也只会变成凭感觉。

**使用分层的评分器组合：**

1. **确定性评分器**，用于任何客观、可由机器检查的内容。它们快速、便宜且稳定，非常适合紧凑的迭代循环和回归关卡（工具调用、JSON 有效性、字符串和模式检查）。

2. **LLM 评分器**，帮助你衡量那些重要但难以纳入确定性规则的方面：正确性、指令遵循、澄清提问是否恰当、完整性和有用性。

3. **音频评分器**，因为用户体验到的是声音，而不是转录文本。音频仍然是最难可靠评判的部分，所以不要等待一个完美的单一评分器，先从简单、可衡量的检查（静音、重叠、打断处理）开始，再随着时间推移逐步叠加更丰富的评分标准。

<p align="center">
<img
  width="900"
  src="../images/realtime_eval_page_09_image_01.png"
  alt="第 9 页的示意图"
/>
</p>


### 6) 评估框架

一个 realtime 评估的可信度取决于运行它的框架。好的框架只有一个任务：**让各次运行具有可比性**。如果同样的输入无法在相同设置下回放并产生相似的结果，就很难进行衡量和迭代。

#### [6.1 从单轮回放开始（“Crawl”框架）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/crawl_harness)

从这里开始。单轮回放能提供最快、最干净的信号，因为你几乎可以固定所有变量。在各次运行之间保持完全相同的音频字节、预处理、VAD 配置、编解码器和分块策略。

实践中，通常最好先关闭语音活动检测（VAD），从而消除一个主要的差异来源。关闭 VAD 后，由你精确决定用户轮次何时结束。

**一个简单的单轮框架如下：**

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_10_image_01.png"
  alt="第 10 页的示意图"
/>
</p>


**更具体地说（用 Realtime API 的术语）：**

1. 生成或加载输入音频

    - 如果数据点是文本，则生成 TTS 音频。
    
        - 通常，从文本 → TTS → 音频开始是最好的第一步，因为它能大幅加快迭代速度。当你可以快速迭代文本时，调整和完善评估会更容易。

2. 将音频流式传入输入缓冲区

    - 以固定大小的分块发送音频（例如：每个分块的帧大小保持一致）。

    - 重要：分块和时序会影响行为。选定一个标准并坚持下去。例如，每个分块 20 ms 能在响应速度和开销之间取得良好平衡。

3. 提交用户音频

    - （推荐）关闭 VAD 时：在最后一个音频分块之后立即提交。

    - 开启 VAD 时：由服务器检测轮次边界。

4. 触发助手响应

    - 关闭 VAD 时：调用 response.create 开始生成。

    - 开启 VAD 时：自动触发。

5. 收集输出

    - 输出音频分块（流式增量）

    - 输出转录文本（如果启用）

    - 工具调用 / 工具参数（如果有）

    - 最终完成事件

6. 评分并持久化
    - 运行评分器

    - 保存结果

#### [6.2 回放已保存的音频（“Walk”框架）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/walk_harness)

当你从合成 TTS 过渡到真实录音时，框架会发生一个重要变化：**你将从已保存的真实音频中流式发送音频缓冲区。**

**对于已保存的音频，流程变为：**

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_10_image_02.png"
  alt="第 10 页的示意图"
/>
</p>


**在实践中如何让评估更贴近真实：**

- **预处理必须与生产环境一致**
  - 相同的重采样、归一化、声道处理、降噪（如有使用）和编码。

  - 将预处理配置与结果一起保存，以便解释分数的变化。

- **流式策略必须明确**

  - 如果你关心延迟：以固定节奏发送分块（例如“每 20ms 发送 20ms 的音频”）。

  - 如果你只关心迭代速度：可以更快地流式发送，但要保持分块大小不变。

- **轮次边界必须可复现**

  - 为了离线可复现性，优先选择关闭 VAD + 手动提交。

  - 如果必须开启 VAD（以匹配生产环境），请记录 VAD 设置并跟踪边界事件，以便调试故障。

#### [6.3 模型模拟的多轮对话（“Run”框架）](https://github.com/openai/openai-cookbook/tree/main/examples/evals/realtime_evals/run_harness)

模型模拟的多轮对话使用一个**用户模拟器**，为完整对话生成下一个用户轮次。它可以提高场景的覆盖面，但前提是各次运行之间的对话片段保持可比性。

**常见循环：**

<p align="center">
<img
  width="1000"
  src="../images/realtime_eval_page_11_image_01.png"
  alt="第 11 页的示意图"
/>
</p>


**模拟的最佳实践：**
- **固定并版本化模拟器提示词：** 像对待代码一样对待它。一个很小的提示词改动对行为的影响可能比更换模型还大。

- **约束随机性：** 固定 temperature 和采样设置。如果可用，使用 seed。在合适的地方使用确定性的轮次（例如用户问候语）。

- **以确定性方式模拟工具：** 为场景定义预期的工具输出 mock，并在助手调用工具时返回这些确切的输出。这能保持环境稳定，让各次运行具有可比性。

- **记录完整轨迹：** 保存每个生成的用户文本轮次以及你流式发送的最终音频字节。持久化工具调用、工具返回值和时间戳。模拟是一台发现引擎。当它发现了真实的失败模式时，就把它回填为一个确定性的脚本化对话片段，用于 crawl 或 walk 方法。

## 第四部分：案例研究

#### 7.1 客服语音机器人

**产品目标与约束**

通过工具快速、安全地解决常见的客服请求。机器人必须收集正确的信息、调用正确的后端操作，并遵守政策。当它无法提供帮助时，必须干净利落地转人工。它必须能应对情绪沮丧的来电者，同时不变得啰嗦或脆弱。

**Crawl、Walk、Run 计划**

**Crawl：合成音频 + 单轮**

聚焦于路由和政策。给定一个简短的请求，机器人应当选择正确的意图、索要缺失的信息，并避免不安全的操作。使用确定性的合成音频，以便快速迭代工具 schema 和提示词。

**Walk：真实音频 + 单轮**

测试在真实采集条件下的理解能力。使用嘈杂环境和类似电话音质下的合成或真实录音。订单号、姓名和地址正是在这种嘈杂音频中容易出错。评估机器人是否会提出澄清问题，而不是靠猜测。

**Run：合成音频 + 多轮模拟**

使用 gpt-realtime 模拟用户并配合工具 mock，模拟完整的工作流：身份验证、账户查询、订单状态、退货资格、退款、创建工单、转人工。加入具有对抗性但贴近真实的模式：来电者中途改变目标、只提供部分信息、在助手说话时插话，或者答非所问。

**人工审查：**

针对预发布（staging）系统进行内部通话测试。这能发现评分器遗漏的用户体验问题：过长的免责声明、重复的提问、身份验证过程中糟糕的轮流发言。

**核心数据集分组和有用的切片**

- 高频意图：订单状态、退货、退款、取消、账单问题、密码重置、预约安排。

- 缺失和冲突的信息：错误的订单号、两个账户、来电者提供昵称、来电者拒绝验证身份。

- 政策边界：超出期限的退货、受限商品、部分退款、订阅取消规则。

- 转人工触发条件：当置信度低或工具出错时，机器人应当转交人工。

- 情绪语气：愤怒、匆忙、困惑。内容目标保持不变，但表达方式很重要。

**使用的评分器**

- 确定性评分器：工具选择、工具参数有效性、必要时的政策用语。

- LLM 评分标准评分器：指令遵循、解决方案的正确性、共情的语气、是否避免了编造政策、是否恰当地转人工，以及是否保持简洁。

- 音频评分器：长时间静音、打断处理。